# Member C: XGBoost + LIME + SHAP Interpretability
## CS667 Project 4: ML Interpretability

**Assigned Tasks:**
- EDA Focus: Target analysis and class balance
- Model: XGBoost with GridSearchCV
- Interpretability: LIME (Task 3_C partial) + SHAP (Task 3_D)
- Task 4: Lead final model comparison

**Prerequisites:** Run `00_data_preparation.ipynb` first!

---
# Part 1: Setup and Data Loading

In [2]:
%pip install xgboost


   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
   --- ------------------------------------ 5.5/72.0 MB 34.2 MB/s eta 0:00:02
   --------- ------------------------------ 17.0/72.0 MB 44.9 MB/s eta 0:00:02
   ----------------- ---------------------- 30.7/72.0 MB 53.5 MB/s eta 0:00:01
   ------------------------- -------------- 45.9/72.0 MB 58.7 MB/s eta 0:00:01
   --------------------------------- ------ 60.0/72.0 MB 62.7 MB/s eta 0:00:01
   ---------------------------------------  71.8/72.0 MB 63.6 MB/s eta 0:00:01
   ---------------------------------------- 72.0/72.0 MB 57.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib

import lime
import lime.lime_tabular

import shap

# Shared constants
RANDOM_STATE = 42
TARGET_COLUMN = 'DEATH_EVENT'

In [4]:
# Load the pre-split data
train_data = pd.read_csv('../data/train.csv')
test_data = pd.read_csv('../data/test.csv')

X_train = train_data.drop(columns=[TARGET_COLUMN])
y_train = train_data[TARGET_COLUMN]
X_test = test_data.drop(columns=[TARGET_COLUMN])
y_test = test_data[TARGET_COLUMN]

feature_names = X_train.columns.tolist()

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {feature_names}")

Training samples: 89
Test samples: 210
Features: ['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time']


---
# Part 2: Exploratory Data Analysis

**Your EDA Focus**: Target variable analysis and class imbalance.

### Investigation Prompts:
> - What is the class distribution of DEATH_EVENT?
> - How might class imbalance affect model training and evaluation?
> - What strategies exist to handle imbalanced data in XGBoost?

In [ ]:
# TODO: Visualize class distribution

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot of class counts
# YOUR CODE HERE

# Pie chart of proportions
# YOUR CODE HERE

plt.tight_layout()
plt.savefig('../visualizations/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# TODO: Calculate class imbalance ratio
# This will be useful for XGBoost's scale_pos_weight parameter

class_counts = y_train.value_counts()
imbalance_ratio = # YOUR CODE HERE

print(f"Class 0 (Survived): {class_counts[0]}")
print(f"Class 1 (Death): {class_counts[1]}")
print(f"Imbalance ratio (neg/pos): {imbalance_ratio:.2f}")

In [ ]:
# TODO: Visualize feature distributions by target class
# Which features show the most separation between classes?

# YOUR CODE HERE

### EDA Findings:

*Document your observations:*

1. Class distribution: ...
2. Imbalance handling strategy: ...
3. Features with clear class separation: ...

---
# Part 3: XGBoost Modeling

### Investigation Prompts:
> - What makes XGBoost different from Random Forest?
> - What do learning_rate, n_estimators, and max_depth control?
> - How can scale_pos_weight help with imbalanced data?
> - What is the relationship between learning_rate and n_estimators?

In [ ]:
# TODO: Define the hyperparameter grid for XGBoost
# Consider: learning_rate, n_estimators, max_depth, subsample, colsample_bytree

xgb_param_grid = {
    # YOUR CODE HERE
    # Hint: 'learning_rate': [0.01, 0.1, 0.2]
    # Hint: 'n_estimators': [50, 100, 200]
    # Hint: 'max_depth': [3, 5, 7]
    # Hint: 'subsample': [0.8, 1.0]
}

print(f"XGBoost Parameter grid: {xgb_param_grid}")

In [ ]:
# TODO: Perform GridSearchCV for XGBoost
# Consider using scale_pos_weight for class imbalance

xgb = XGBClassifier(
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric='logloss',
    # scale_pos_weight=imbalance_ratio  # Optional: helps with imbalance
)

xgb_grid_search = GridSearchCV(
    # YOUR CODE HERE
)

# Fit
# YOUR CODE HERE

print(f"Best XGBoost parameters: {xgb_grid_search.best_params_}")
print(f"Best XGBoost CV AUC-ROC: {xgb_grid_search.best_score_:.4f}")

In [ ]:
# Evaluate XGBoost on test set
best_xgb = xgb_grid_search.best_estimator_

y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_auc_roc = roc_auc_score(y_test, y_proba_xgb)

print(f"\n=== XGBoost Results ===")
print(f"Test Accuracy: {xgb_accuracy:.4f}")
print(f"Test AUC-ROC: {xgb_auc_roc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# Save XGBoost model
joblib.dump(best_xgb, '../models/xgboost_best.pkl')
print("XGBoost model saved.")

---
# Part 4: LIME for XGBoost (Task 3_C partial)

## LIME Investigation Prompts

> **Research these questions:**
> 
> 1. How does LIME create local explanations for any model?
> 2. What do the coefficients represent in LIME's local linear model?
> 3. What does R² tell you about the quality of the explanation?
> 4. How might LIME explanations for XGBoost differ from Random Forest?

In [ ]:
# TODO: Create a LimeTabularExplainer

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    # YOUR CODE HERE
)

print("LIME explainer created.")

In [ ]:
# Find example indices
positive_idx = y_test[y_test == 1].index[0]
negative_idx = y_test[y_test == 0].index[0]

print(f"Positive example index: {positive_idx}")
print(f"Negative example index: {negative_idx}")

In [ ]:
# TODO: Explain POSITIVE example with LIME

pos_idx_loc = list(X_test.index).index(positive_idx)
pos_instance = X_test.iloc[pos_idx_loc].values

print("=== LIME Explanation for POSITIVE Example ===")
# YOUR CODE HERE

In [ ]:
# TODO: Access LIME model details (coefficients, intercept, R²)
# As required by the project

print("\nLIME Linear Model Details:")
# YOUR CODE HERE

In [ ]:
# TODO: Explain NEGATIVE example with LIME

print("=== LIME Explanation for NEGATIVE Example ===")
# YOUR CODE HERE

---
# Part 5: SHAP for XGBoost (Task 3_D)

## SHAP Investigation Prompts

> **Research these questions (this is key for deep understanding):**
> 
> 1. What are Shapley values from game theory? Why are they considered "fair"?
> 2. Why is `TreeExplainer` specifically designed for tree-based models?
> 3. What is `explainer.expected_value` and why is it the "base value"?
> 4. In a force_plot:
>    - What do red arrows represent?
>    - What do blue arrows represent?
>    - How do you read the final prediction?
> 5. In a summary_plot:
>    - What does the x-axis show?
>    - What does the y-axis show?
>    - What does the color represent?
> 6. How do SHAP values differ from LIME coefficients?

### 5.1 Create SHAP TreeExplainer

In [ ]:
# TODO: Create a SHAP TreeExplainer for XGBoost
# Hint: shap.TreeExplainer(model)

shap_explainer = # YOUR CODE HERE

# Get the base value (expected_value)
base_value = shap_explainer.expected_value
print(f"Base value (expected_value): {base_value}")

In [ ]:
# TODO: Calculate SHAP values for test set
# Hint: shap_values = shap_explainer.shap_values(X_test)

shap_values = # YOUR CODE HERE

print(f"SHAP values shape: {shap_values.shape}")

### 5.2 Force Plots for Individual Predictions

Create force_plot visualizations for one positive and one negative example.

In [ ]:
# Initialize SHAP JS visualization (needed for notebook display)
shap.initjs()

In [ ]:
# TODO: Create force_plot for POSITIVE example
# Hint: shap.force_plot(base_value, shap_values[idx], X_test.iloc[idx])

pos_idx_loc = list(X_test.index).index(positive_idx)

print("=== SHAP Force Plot for POSITIVE Example ===")
# YOUR CODE HERE

In [ ]:
# TODO: Create force_plot for NEGATIVE example

neg_idx_loc = list(X_test.index).index(negative_idx)

print("=== SHAP Force Plot for NEGATIVE Example ===")
# YOUR CODE HERE

### 5.3 Summary Plots

Create summary_plot for each class/label as required.

In [ ]:
# TODO: Create summary_plot showing feature importance
# Hint: shap.summary_plot(shap_values, X_test)

print("=== SHAP Summary Plot (Feature Importance) ===")
# YOUR CODE HERE

plt.savefig('../visualizations/shap_summary.png', dpi=150, bbox_inches='tight')

In [ ]:
# TODO: Create summary_plot with plot_type='bar' for clearer importance ranking

print("=== SHAP Feature Importance Bar Plot ===")
# YOUR CODE HERE

plt.savefig('../visualizations/shap_importance_bar.png', dpi=150, bbox_inches='tight')

### 5.4 SHAP Interpretation Summary

*Document your SHAP findings:*

1. **Base value meaning:** ...

2. **Top features by SHAP importance:**
   - Feature 1: ...
   - Feature 2: ...
   - Feature 3: ...

3. **Positive example force_plot interpretation:**
   - Features pushing toward death (red): ...
   - Features pushing toward survival (blue): ...

4. **Negative example force_plot interpretation:**
   - Features pushing toward death: ...
   - Features pushing toward survival: ...

5. **Comparison SHAP vs LIME:**
   - Similarities: ...
   - Differences: ...
   - Which do you trust more for this model? Why?

---
# Part 6: Task 4 - Model Comparison (You Lead This)

Load all team models and create the final comparison.

In [ ]:
# TODO: Fill in the agreed indices from team discussion
POSITIVE_EXAMPLE_IDX = None  # Update with team-agreed index
NEGATIVE_EXAMPLE_IDX = None  # Update with team-agreed index

if POSITIVE_EXAMPLE_IDX is None:
    print("WARNING: Update example indices!")

In [ ]:
# Load all team models (run after team members save their models)
try:
    lr_model = joblib.load('../models/logistic_regression_best.pkl')
    lr_scaler = joblib.load('../models/lr_scaler.pkl')
    dt_model = joblib.load('../models/decision_tree_best.pkl')
    rf_model = joblib.load('../models/random_forest_best.pkl')
    xgb_model = best_xgb  # Already loaded
    print("All models loaded successfully!")
except FileNotFoundError as e:
    print(f"Model not found: {e}")
    print("Make sure all team members have saved their models.")

In [ ]:
# TODO: Create the comparison table as specified in the project
# Format:
# False/True label: 0/1
# * LR: [prob_T prob_F]
# * DT: [prob_T prob_F]
# * RF: [prob_T prob_F]
# * XGB: [prob_T prob_F]

def compare_all_models(idx, true_label):
    """Compare predictions from all 4 models."""
    pos = list(X_test.index).index(idx)
    sample = X_test.iloc[pos:pos+1]
    
    # Scale for LR
    sample_scaled = lr_scaler.transform(sample)
    
    print(f"True Label: {true_label}")
    print("-" * 40)
    
    # LR
    lr_proba = lr_model.predict_proba(sample_scaled)[0]
    print(f"* LR:  [P(0)={lr_proba[0]:.4f}  P(1)={lr_proba[1]:.4f}]")
    
    # DT
    dt_proba = dt_model.predict_proba(sample)[0]
    print(f"* DT:  [P(0)={dt_proba[0]:.4f}  P(1)={dt_proba[1]:.4f}]")
    
    # RF
    rf_proba = rf_model.predict_proba(sample)[0]
    print(f"* RF:  [P(0)={rf_proba[0]:.4f}  P(1)={rf_proba[1]:.4f}]")
    
    # XGB
    xgb_proba = xgb_model.predict_proba(sample)[0]
    print(f"* XGB: [P(0)={xgb_proba[0]:.4f}  P(1)={xgb_proba[1]:.4f}]")
    
    # Determine best prediction
    predictions = {
        'LR': lr_proba[true_label],
        'DT': dt_proba[true_label],
        'RF': rf_proba[true_label],
        'XGB': xgb_proba[true_label]
    }
    best_model = max(predictions, key=predictions.get)
    print(f"\nBest prediction confidence: {best_model} ({predictions[best_model]:.4f})")

if POSITIVE_EXAMPLE_IDX is not None:
    print("\n" + "="*50)
    print("POSITIVE EXAMPLE (Death)")
    print("="*50)
    compare_all_models(POSITIVE_EXAMPLE_IDX, 1)
    
    print("\n" + "="*50)
    print("NEGATIVE EXAMPLE (Survived)")
    print("="*50)
    compare_all_models(NEGATIVE_EXAMPLE_IDX, 0)

---
# Part 7: Summary for Merge

### XGBoost Results:

| Metric | Value |
|--------|-------|
| Best Hyperparameters | ... |
| Test Accuracy | ... |
| Test AUC-ROC | ... |

### Files Saved:
- `../models/xgboost_best.pkl`
- `../visualizations/class_distribution.png`
- `../visualizations/shap_summary.png`
- `../visualizations/shap_importance_bar.png`

### Key Interpretability Findings:

**LIME:**
1. ...
2. R² quality: ...

**SHAP:**
1. Top features: ...
2. Key insights from force plots: ...
3. SHAP vs LIME comparison: ...

### Task 4 Comparison Summary:
- Best overall model: ...
- Reasoning: ...